In [20]:
#Day 4, Topic 7: Grouping with .value_counts() and .nunique()

import pandas as pd
df = pd.read_csv('titanic.csv')

In [21]:
#.value_counts() on Grouped Data
sex_counts = df.groupby('Pclass')['Sex'].value_counts()
sex_counts

Pclass  Sex   
1       male      122
        female     94
2       male      108
        female     76
3       male      347
        female    144
Name: count, dtype: int64

In [22]:
#Flatten for readability:
sex_counts.reset_index(name='count')

,Pclass,Sex,count
0,1,male,122
1,1,female,94
2,2,male,108
3,2,female,76
4,3,male,347
5,3,female,144


In [23]:
#Normalized .value_counts() for Proportions
sex_pct = df.groupby('Pclass')['Sex'].value_counts(normalize=True)
sex_pct

Pclass  Sex   
1       male      0.564815
        female    0.435185
2       male      0.586957
        female    0.413043
3       male      0.706721
        female    0.293279
Name: proportion, dtype: float64

In [24]:
#.nunique() – Count Unique Values per Group
unique_tickets = df.groupby('Pclass')['Ticket'].nunique()
unique_tickets

Pclass
1    147
2    140
3    394
Name: Ticket, dtype: int64

In [25]:
df.groupby('Embarked').agg(
    unique_passengers=('PassengerId', 'nunique'),
    unique_names=('Name', 'nunique'),
    unique_cabins=('Cabin', 'nunique')
)

,unique_passengers,unique_names,unique_cabins
Embarked,,,
C,168,168,57
Q,77,77,3
S,644,644,89


In [26]:
#Combining .value_counts() with .unstack() for Cross‑Tabulation
sex_table = df.groupby('Pclass')['Sex'].value_counts().unstack()
sex_table

Sex,female,male
Pclass,,
1,94,122
2,76,108
3,144,347


In [27]:
#.value_counts() on Multiple Columns Simultaneously
combo_counts = df[['Pclass', 'Sex', 'Survived']].value_counts().reset_index(name='count')
combo_counts.head()

,Pclass,Sex,Survived,count
0,3,male,0,300
1,1,female,1,91
2,2,male,0,91
3,1,male,0,77
4,3,female,0,72


In [28]:
#Practice Activity: .value_counts() and .nunique()
import pandas as pd

df = pd.read_csv('titanic.csv')

In [29]:
"""Task 1: Use .groupby() and .value_counts() to count how many passengers survived vs. did not survive within each Pclass. 
Flatten the result with .reset_index()"""

passen = df.groupby('Pclass')['Survived'].value_counts().reset_index(name='counts')
passen

,Pclass,Survived,counts
0,1,1,136
1,1,0,80
2,2,0,97
3,2,1,87
4,3,0,372
5,3,1,119


In [30]:
"""Task 2: Repeat Task 1, but use normalize=True to get survival proportions per class. 
Which class had the highest survival rate?"""

prop = df.groupby('Pclass')['Survived'].value_counts(normalize=True).reset_index(name='Percentage')

prop

,Pclass,Survived,Percentage
0,1,1,0.629630
1,1,0,0.370370
2,2,0,0.527174
3,2,1,0.472826
4,3,0,0.757637
5,3,1,0.242363


In [31]:
"""Task 3: Use .groupby() and .nunique() to find out how many unique ticket numbers exist for each 
combination of Embarked and Pclass. Flatten the result."""

result = df.groupby(['Embarked', 'Pclass'])['Ticket'].nunique().reset_index()
result

,Embarked,Pclass,Ticket
0,C,1,57
1,C,2,12
2,C,3,53
3,Q,1,1
4,Q,2,3
5,Q,3,62
6,S,1,90
7,S,2,125
8,S,3,279


In [32]:
#Task 4: For each Embarked port, count how many unique passenger names exist. 
# (Hint: group by Embarked, then use .nunique() on Name.)

result = df.groupby('Embarked')['Name'].nunique()

result

Embarked
C    168
Q     77
S    644
Name: Name, dtype: int64

In [33]:
"""Task 5 (Bonus): Create a contingency table showing the count of passengers by Pclass (rows) and Embarked (columns) 
using .groupby() and .value_counts() with .unstack(). Then, add a "Total" column summing each row."""

contingency_table = df.groupby('Pclass')['Embarked'].value_counts().unstack()
contingency_table['Total'] = contingency_table.sum(axis=1)
contingency_table

Embarked,C,Q,S,Total
Pclass,,,,
1,85,2,127,214
2,17,3,164,184
3,66,72,353,491


In [34]:
#Day 4, Topic 8: Pivot Tables
import pandas as pd
df = pd.read_csv('titanic.csv')

In [35]:
#pd.pivot_table() Basics
pivot = pd.pivot_table(df, values='Fare', index='Pclass', columns='Sex', aggfunc='mean')

pivot

Sex,female,male
Pclass,,
1,106.125798,67.226127
2,21.970121,19.741782
3,16.118810,12.661633


In [36]:
#Multiple Values and Aggregations
pivot = pd.pivot_table(df,
                       values=['Age', 'Fare'],
                       index='Pclass',
                       columns='Sex',
                       aggfunc=['mean', 'median'])

pivot

mean                                   median                  \
              Age                   Fare               Age            Fare   
Sex        female       male      female       male female  male    female   
Pclass                                                                       
1       34.611765  41.281386  106.125798  67.226127   35.0  40.0  82.66455   
2       28.722973  30.740707   21.970121  19.741782   28.0  30.0  22.00000   
3       21.750000  26.507589   16.118810  12.661633   21.5  25.0  12.47500   

                 
                 
Sex        male  
Pclass           
1       41.2625  
2       13.0000  
3        7.9250

In [37]:
#Adding Margins (Totals)
pivot = pd.pivot_table(df,
                       values='Survived',
                       index='Pclass',
                       columns='Sex',
                       aggfunc='mean',
                       margins=True,
                       margins_name='Total')

pivot

Sex,female,male,Total
Pclass,,,
1,0.968085,0.368852,0.629630
2,0.921053,0.157407,0.472826
3,0.500000,0.135447,0.242363
Total,0.742038,0.188908,0.383838


In [38]:
#Handling Missing Combinations with fill_value
pivot = pd.pivot_table(df,
                       values='Survived',
                       index='Pclass',
                       columns='Embarked',
                       aggfunc='count',
                       fill_value=0)

pivot

Embarked,C,Q,S
Pclass,,,
1,85,2,127
2,17,3,164
3,66,72,353


In [39]:
#Grouping by Multiple Index/Columns
pivot = pd.pivot_table(df, 
                       values='Fare',
                       index=['Pclass', 'Sex'],
                       columns='Embarked',
                       aggfunc='mean')

pivot

Embarked                C          Q          S
Pclass Sex                                     
1      female  115.640309  90.000000  99.026910
       male     93.536707  90.000000  52.949947
2      female   25.268457  12.350000  21.912687
       male     25.421250  12.350000  19.232474
3      female   14.694926  10.307833  18.670077
       male      9.352237  11.924251  13.307149

In [40]:
#Practice Activity: Pivot Tables
import pandas as pd
import numpy as np

df = pd.read_csv('titanic.csv')

In [41]:
"""Task 1: Create a pivot table showing the average Age for each 
combination of Pclass (rows) and Sex (columns). Use pd.pivot_table()."""

pivot = pd.pivot_table(df,
                       values='Age',
                       index='Pclass',
                       columns='Sex',
                       aggfunc='mean')

pivot

Sex,female,male
Pclass,,
1,34.611765,41.281386
2,28.722973,30.740707
3,21.750000,26.507589


In [42]:
"""Task 2: Create a pivot table showing the median Fare and mean Survived for each Embarked (rows) and Pclass 
(columns). Use a list for values and a list for aggfunc."""

pivot = pd.pivot_table(df, 
                       values=['Fare', 'Survived'],
                       index='Embarked',
                       columns='Pclass',
                       aggfunc={'Fare': 'median', 'Survived': 'mean'})

pivot

Fare                 Survived                    
Pclass          1      2       3         1         2         3
Embarked                                                      
C         78.2667  24.00  7.8958  0.694118  0.529412  0.378788
Q         90.0000  12.35  7.7500  0.500000  0.666667  0.375000
S         52.0000  13.50  8.0500  0.582677  0.463415  0.189802

In [43]:
"""Task 3: Create a pivot table showing the count of passengers (PassengerId) for each Pclass (rows) and 
Embarked (columns). Add margins to show row and column totals. Fill missing combinations with 0."""

pivot = pd.pivot_table(df, 
                       values='PassengerId',
                       index='Pclass',
                       columns='Embarked',
                       aggfunc='count',
                       fill_value=0,
                       margins=True,
                       margins_name='Total')

pivot

Embarked,C,Q,S,Total
Pclass,,,,
1,85,2,127,214
2,17,3,164,184
3,66,72,353,491
Total,168,77,644,889


In [44]:
"""Task 4: Create a pivot table with index=['Pclass', 'Sex'] and columns='Embarked', 
showing the maximum Fare per combination.
"""

pivot = pd.pivot_table(df, 
                       values='Fare',
                       index=['Pclass', 'Sex'],
                       columns='Embarked',
                       aggfunc='max')

pivot

Embarked              C       Q       S
Pclass Sex                             
1      female  512.3292  90.000  263.00
       male    512.3292  90.000  263.00
2      female   41.5792  12.350   65.00
       male     41.5792  12.350   73.50
3      female   22.3583  29.125   69.55
       male     21.6792  29.125   69.55

In [45]:
"""Task 5 (Bonus): Create a pivot table showing the survival rate (mean of Survived) 
for each Pclass (rows) and AgeGroup (columns). First, create AgeGroup using pd.cut() 
with bins [0,18,35,60,100] and labels ['Child','Young Adult','Adult','Senior']. 
Then pivot with margins.
"""

df['AgeGroup'] = pd.cut(df['Age'], bins=[0, 18, 35, 60, 100], labels=['Child', 'Young Adult', 'Adult', 'Senior'])

pivot = pd.pivot_table(df,
                       values='Survived',
                       index='Pclass',
                       columns='AgeGroup',
                       aggfunc='mean',
                       margins=True,
                       margins_name='Total')

pivot

C:\Users\Faizan Toheed\AppData\Local\Temp\ipykernel_14256\2336470792.py:9: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  pivot = pd.pivot_table(df,


AgeGroup,Child,Young Adult,Adult,Senior,Total
Pclass,,,,,
1,0.875000,0.757576,0.611111,0.214286,0.655914
2,0.793103,0.436170,0.382979,0.333333,0.479769
3,0.351064,0.232323,0.086207,0.200000,0.239437
Total,0.503597,0.382682,0.400000,0.227273,0.406162
